# Model Tuning

The point of this notebook is the find the best mean model according to out-of-sample log-likelihood and then also compare performance of GARCH models according to the same metric.

## Setup

In [1]:
import os
from pathlib import Path

import pandas as pd

In [2]:
# Find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# Set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: C:\Users\couch\OneDrive\Assignments\Master Thesis\Repo


In [3]:
from src.config import EVAL_MEAN_MODELS, EVAL_VOLATILITY_MODELS, VAL_END, VAL_START, WINDOW_SIZE
from src.simulation import run_backtest

## Mean Model Tuning

First, it is wise to pick one mean model according to the out-of-sample log-likelihood. Naturally, the naive covariance model is used here.

In [4]:
df = pd.read_parquet("results/processed_data/stock_data.parquet", engine="pyarrow")
df.index = pd.to_datetime(df.index)
df = df[df.index <= VAL_END]
df

,ALR,CDR,CPS,DNP,EBP,JSW,KGH,LPP,LTS,MBK,MDV,OPL,PEO,PGE,PGN,PKN,PKO,PLY,PZU,TPE
Date,,,,,,,,,,,,,,,,,,,,
2014-01-03,-0.009629,-0.006342,-0.001011,NaN,-0.010414,0.009294,-0.003378,0.011834,-0.007074,-0.017298,-0.004692,0.005099,-0.010880,0.001238,-0.021053,-0.014708,-0.015544,NaN,-0.016329,-0.013668
2014-01-07,-0.019666,-0.017503,-0.023013,NaN,-0.034886,-0.023588,-0.021377,-0.030703,-0.030714,-0.028988,-0.017684,-0.008172,-0.029895,-0.009326,-0.047534,-0.009927,-0.015267,NaN,-0.019091,-0.025553
2014-01-08,0.009443,-0.005311,-0.026207,NaN,0.018792,0.002082,0.001296,-0.006255,-0.009915,0.024385,-0.019777,-0.005141,0.031297,-0.007524,-0.008147,-0.002378,0.001303,NaN,0.006554,0.007034
2014-01-09,0.000000,-0.003557,-0.005325,NaN,0.000000,-0.051194,-0.032013,-0.030700,-0.001608,-0.034839,0.011035,0.000000,-0.031297,0.008772,0.020244,-0.014389,-0.010734,NaN,-0.024862,-0.016490
2014-01-10,0.000000,-0.002378,0.014312,NaN,-0.008011,-0.017259,-0.005362,0.000000,-0.002902,0.018718,-0.016822,0.008214,0.022858,-0.018257,-0.028457,0.019139,0.008388,NaN,-0.016298,-0.004762
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-19,-0.008316,0.002235,-0.035718,0.015839,-0.010724,-0.009950,0.010417,0.002861,-0.006439,0.011233,0.011215,-0.005450,0.003980,-0.011187,0.009479,0.012261,-0.007685,-0.011856,0.007701,-0.024244
2019-12-20,-0.015428,0.005197,0.014441,0.014185,0.000674,-0.009545,-0.015456,-0.003434,0.028654,-0.004799,-0.009337,-0.022100,-0.005976,-0.006270,-0.007101,-0.014637,-0.006881,0.017145,0.004444,-0.018576
2019-12-23,-0.009943,0.035997,0.002148,-0.004942,0.014706,0.010545,-0.000211,0.003434,0.017945,0.040843,0.007477,-0.015482,0.011423,0.002513,0.007101,0.012290,-0.002593,-0.012980,-0.006426,-0.006270


In [5]:
for model in EVAL_MEAN_MODELS:
    save_path = Path(f"results/mean_tuning/data/{model}.parquet")
    log_path = Path(f"results/mean_tuning/logs/{model}.txt")

    # Skip if parquet output already exists
    if save_path.exists():
        print(f"Skipping '{model}': file already exists at {save_path}")
        continue

    print(f"Running backtest for MeanModel: {model}")

    run_backtest(
        returns_df=df,
        window_size=WINDOW_SIZE,
        start_date=VAL_START,
        end_date=VAL_END,
        mean_model=model,
        volatility_model="naive",
        optimize_portfolio_flag=False,
        save_path=save_path,
        ga_metric=None,
        log_path=log_path,
    )

Skipping 'naive': file already exists at results\mean_tuning\data\naive.parquet
Running backtest for MeanModel: ar


Running backtest: 100%|██████████| 495/495 [01:20<00:00,  6.16it/s]

Skipping 'var': file already exists at results\mean_tuning\data\var.parquet
Skipping 'var_lasso': file already exists at results\mean_tuning\data\var_lasso.parquet


In [6]:
data_dir = Path("results/mean_tuning/data")
log_likelihoods = {}

# Iterate over all parquet files in the results directory
for file_path in sorted(data_dir.glob("*.parquet")):
    model_name = file_path.stem.upper()  # Extracts model name from filename
    df_model = pd.read_parquet(file_path)
    log_likelihoods[model_name] = df_model["log_likelihood"]

# Combine into a single DataFrame indexed by date
mean_ll_df = pd.DataFrame(log_likelihoods).sort_index()
mean_ll_df

,AR,NAIVE,VAR,VAR_LASSO
date,,,,
2018-01-03,41.259633,40.314141,40.300917,40.223188
2018-01-04,37.779267,38.110022,38.090146,38.124530
2018-01-05,47.508619,45.747043,45.766818,45.786745
2018-01-08,50.284747,50.574090,50.583955,50.614025
2018-01-09,48.999423,48.601722,48.621028,48.680946
...,...,...,...,...
2019-12-19,56.200425,55.750973,55.775633,55.766981
2019-12-20,56.984471,56.451958,56.449685,56.493783
2019-12-23,55.174805,55.040268,55.052966,54.890605


In [7]:
mean_ll_df.isna().sum()

AR           0
NAIVE        0
VAR          0
VAR_LASSO    0
dtype: int64

In [8]:
# Calculate per-column sum (total log-likelihood for each model)
total_ll = mean_ll_df.sum()

# Build formatted summary DataFrame with model names as index
summary_mean_ll_df = pd.DataFrame(
    {"Log-wiarygodność": [f"{val:.4f}".replace(".", ",") for val in total_ll]},
    index=total_ll.index,
)
summary_mean_ll_df

,Log-wiarygodność
AR,"23750,7053"
NAIVE,"23787,6679"
VAR,"23786,9421"
VAR_LASSO,"23789,5071"


## Variance Model Tuning

Now only using the best mean model, multiple variance models should be fitted to get an idea regarding their performance on this dataset and see if there are not any problems with estimation.

In [9]:
for model in EVAL_VOLATILITY_MODELS:
    save_path = Path(f"results/volatility_tuning/data/{model}.parquet")
    log_path = Path(f"results/volatility_tuning/logs/{model}.txt")

    # Skip if parquet output already exists
    if save_path.exists():
        print(f"Skipping '{model}': file already exists at {save_path}")
        continue

    print(f"Running backtest for VolatilityModel: {model}")

    run_backtest(
        returns_df=df,
        window_size=WINDOW_SIZE,
        start_date=VAL_START,
        end_date=VAL_END,
        mean_model="var_lasso",
        volatility_model=model,
        optimize_portfolio_flag=False,
        save_path=save_path,
        ga_metric=None,
        log_path=log_path,
    )

Skipping 'naive': file already exists at results\volatility_tuning\data\naive.parquet
Skipping 'ccc': file already exists at results\volatility_tuning\data\ccc.parquet
Skipping 'dcc': file already exists at results\volatility_tuning\data\dcc.parquet
Running backtest for VolatilityModel: go_garch


Running backtest: 100%|██████████| 495/495 [2:33:36<00:00, 18.62s/it]  


In [10]:
data_dir = Path("results/volatility_tuning/data")
log_likelihoods = {}

# Iterate over all parquet files in the results directory
for file_path in sorted(data_dir.glob("*.parquet")):
    model_name = file_path.stem.upper()  # Extracts model name from filename
    df_model = pd.read_parquet(file_path)
    log_likelihoods[model_name] = df_model["log_likelihood"]

# Combine into a single DataFrame indexed by date
volatility_ll_df = pd.DataFrame(log_likelihoods).sort_index()
volatility_ll_df

,CCC,DCC,GO_GARCH,NAIVE
date,,,,
2018-01-03,37.242720,37.799371,39.133887,40.223188
2018-01-04,30.301854,34.215946,36.861709,38.124530
2018-01-05,48.041466,47.424845,45.298164,45.786745
2018-01-08,51.386196,51.216436,51.224227,50.614025
2018-01-09,50.032860,49.902284,49.900466,48.680946
...,...,...,...,...
2019-12-19,56.825683,56.722196,55.827378,55.766981
2019-12-20,57.660687,57.372835,56.770027,56.493783
2019-12-23,53.812125,53.325583,55.265458,54.890605


In [11]:
volatility_ll_df.isna().sum()

CCC         0
DCC         0
GO_GARCH    0
NAIVE       0
dtype: int64

In [12]:
# Calculate per-column sum (total log-likelihood for each model)
total_ll = volatility_ll_df.sum()

# Build formatted summary DataFrame with model names as index
summary_volatility_ll_df = pd.DataFrame(
    {"Log-wiarygodność": [f"{val:.4f}".replace(".", ",") for val in total_ll]},
    index=total_ll.index,
)
summary_volatility_ll_df

,Log-wiarygodność
CCC,"23828,6167"
DCC,"23858,4636"
GO_GARCH,"23803,4752"
NAIVE,"23789,5071"
